# 11_full_recourse_expansion

Population-wide counterfactual recourse for all at-risk individuals.

DiCE's generic search (random / genetic / kdtree) is slow per instance and
stalls on some cases, so full-population expansion was previously deferred. But
the guardrails constrain the actionable space to just four variables with fixed
directions (BMI reduction only, quit smoking, start exercise, more walking).
That small, direction-constrained space lets us enumerate guardrail-compliant
candidates on a grid and score them all in a single vectorized batch - the
entire top-30% risk group is processed in ~1 second.

Outputs: per-person recourse feasibility, the distribution of required BMI
reduction, recourse availability by option type (Table 14), and feasibility by
age group (Figure 6). The dominant finding: recourse feasibility falls sharply
with age (64% at 45-54 down to ~0% at 75+), because age (immutable) drives the
residual risk - the same non-monotonic recourse pattern seen in notebooks 03
and 08, now quantified at population scale.

In [1]:
# 11_full_recourse_expansion.ipynb
# Population-wide, guardrail-constrained counterfactual recourse via batch scoring.

import os
import time
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = os.path.abspath("..")
DATA = os.path.join(ROOT, "data")
FIG  = os.path.join(ROOT, "results", "figures")
TAB  = os.path.join(ROOT, "results", "tables")

htn = pd.read_parquet(os.path.join(DATA, "htn_analysis.parquet"))
htn["female"] = (htn["SEX"] == 2).astype(float)
FEATS = ["BMI", "age", "female", "smoke_cur", "exer_reg", "walk_days"]
X = htn[FEATS].astype(float); y = htn["incident"].astype(int)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
clf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=30,
        class_weight="balanced", random_state=42).fit(Xtr, ytr)

allp   = clf.predict_proba(X)[:, 1]
target = np.percentile(allp, 50)     # cohort median risk = recourse target
Xv     = X.values
print(f"Target risk line (median): {target:.4f}")

Target risk line (median): 0.4497


In [2]:
# Enumerate guardrail-compliant counterfactual candidates for one person.
BMI_REDS = np.arange(0, 0.155, 0.01)   # 0-15% relative BMI reduction, 1% steps

def enumerate_cf(row):
    b0, age, fem, smk, exr, wlk = row
    out = []
    # single: BMI reduction (floor 18.5)
    for r in BMI_REDS:
        nb = max(b0 * (1 - r), 18.5)
        out.append((nb, age, fem, smk, exr, wlk, "BMI only", abs(b0 - nb)))
    # single: behaviour (only in the improving direction)
    if smk == 1:
        out.append((b0, age, fem, 0, exr, wlk, "Quit smoking", 0))
    if exr == 0:
        out.append((b0, age, fem, smk, 1, wlk, "Start exercise", 0))
    # combinations: BMI + behaviour
    for r in BMI_REDS[1:]:
        nb = max(b0 * (1 - r), 18.5)
        if smk == 1:
            out.append((nb, age, fem, 0, exr, wlk, "BMI + quit", abs(b0 - nb)))
        if exr == 0:
            out.append((nb, age, fem, smk, 1, wlk, "BMI + exercise", abs(b0 - nb)))
    return out

In [3]:
# Batch-score every candidate for the whole top-30% risk group at once.
atrisk_idx = np.where(allp >= np.percentile(allp, 70))[0]

cands, owner, typ, bmi_kg = [], [], [], []
for i in atrisk_idx:
    for c in enumerate_cf(Xv[i]):
        cands.append(c[:6]); owner.append(i); typ.append(c[6]); bmi_kg.append(c[7])
cands = np.array(cands)

t0 = time.time()
preds = clf.predict_proba(cands)[:, 1]      # single vectorized prediction
print(f"Scored {len(cands)} candidates for {len(atrisk_idx)} people "
      f"in {time.time()-t0:.1f}s")

R = pd.DataFrame({"owner": owner, "type": typ, "bmi_kg": bmi_kg, "pred": preds})
R["ok"] = R["pred"] <= target

Scored 236224 candidates for 9354 people in 1.2s


In [4]:
# Feasibility summary and required-effort distribution (Table 13).
n_target = len(atrisk_idx)
n_solved = R.loc[R["ok"], "owner"].nunique()
ok_bmi = R[(R.ok) & (R.type == "BMI only")].groupby("owner")["bmi_kg"].min()

print(f"At-risk (top 30%)      : {n_target}")
print(f"Recourse feasible      : {n_solved} ({100*n_solved/n_target:.1f}%)")
print(f"Infeasible (age-dom.)  : {n_target-n_solved} ({100*(n_target-n_solved)/n_target:.1f}%)")
print(f"Median BMI reduction   : {ok_bmi.median():.1f} kg/m2")

pd.DataFrame({
    "Metric": ["At-risk (top 30%)", "Recourse feasible", "Infeasible (age-dominated)",
               "Median BMI reduction (kg/m2)", "Mean BMI reduction"],
    "Value":  [n_target, n_solved, n_target - n_solved,
               round(ok_bmi.median(), 1), round(ok_bmi.mean(), 1)],
}).to_csv(os.path.join(TAB, "table13_full_recourse.csv"), index=False)

avail = R[R.ok].groupby("type")["owner"].nunique().sort_values(ascending=False)
avail.to_csv(os.path.join(TAB, "table14_recourse_by_type.csv"), header=["n_feasible"])
print("\nRecourse availability by option type:")
print(avail.to_string())

At-risk (top 30%)      : 9354
Recourse feasible      : 1783 (19.1%)
Infeasible (age-dom.)  : 7571 (80.9%)
Median BMI reduction   : 2.6 kg/m2

Recourse availability by option type:
type
BMI only          1078
BMI + exercise     813
BMI + quit         273
Start exercise       8
Quit smoking         2


In [5]:
# Per-person export + figures.
person = pd.DataFrame({"owner": atrisk_idx})
person["feasible"] = person["owner"].isin(R.loc[R.ok, "owner"]).astype(int)
person["risk"] = allp[atrisk_idx]
person["age"]  = Xv[atrisk_idx, 1]
person["BMI"]  = Xv[atrisk_idx, 0]
person.to_parquet(os.path.join(DATA, "full_recourse.parquet"))

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size": 11, "axes.edgecolor": "0.3",
                     "grid.color": "0.85", "savefig.dpi": 600})

# Figure 6: recourse feasibility by age group.
person["age_grp"] = pd.cut(person["age"], [19,45,55,65,75,120],
                           labels=["19-44","45-54","55-64","65-74","75+"])
feas = person.groupby("age_grp", observed=True)["feasible"].mean() * 100
fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.bar(range(len(feas)), feas.values, color="0.4", edgecolor="0.2", width=0.65)
ax.set_xticks(range(len(feas))); ax.set_xticklabels(feas.index)
ax.set_xlabel("Age group"); ax.set_ylabel("Recourse-feasible (%)"); ax.set_ylim(0, 100)
fig.savefig(os.path.join(FIG, "fig6_recourse_feasibility_by_age.png"), dpi=600, bbox_inches="tight")
fig.savefig(os.path.join(FIG, "fig6_recourse_feasibility_by_age.pdf"), bbox_inches="tight")
plt.close(fig)

# Figure 7: recourse availability by option type.
a = avail.sort_values()
grays = ["0.7","0.6","0.5","0.35","0.2"][:len(a)]
fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.barh(range(len(a)), a.values, color=grays, edgecolor="0.2")
ax.set_yticks(range(len(a))); ax.set_yticklabels(a.index)
ax.set_xlabel("Individuals with feasible recourse")
fig.savefig(os.path.join(FIG, "fig7_recourse_by_type.png"), dpi=600, bbox_inches="tight")
fig.savefig(os.path.join(FIG, "fig7_recourse_by_type.pdf"), bbox_inches="tight")
plt.close(fig)
print("Figures 6 and 7 saved (png + pdf).")

Figures 6 and 7 saved (png + pdf).
